# Xây dựng Ứng dụng Tạo Hình ảnh (OpenAI)

Có nhiều hơn thế về LLMs ngoài việc tạo văn bản. Bạn cũng có thể tạo hình ảnh từ các mô tả văn bản. Hình ảnh như một phương thức hữu ích trong các lĩnh vực MedTech, kiến trúc, du lịch, phát triển trò chơi, marketing và nhiều hơn nữa. Trong bài học này, chúng ta làm việc với các mô hình **GPT Image** hiện đại trên nền tảng OpenAI.

## Mục tiêu học tập

Kết thúc bài học này, bạn sẽ có thể:

- Giải thích tạo hình ảnh là gì và nó hữu ích ở đâu.
- Hiểu về gia đình mô hình `gpt-image` và cách nó khác với các mô hình DALL·E cũ.
- Xây dựng ứng dụng tạo hình ảnh và chỉnh sửa hình ảnh.

## Tạo ảnh là gì?

Các mô hình tạo ảnh tạo ra hình ảnh từ một lời nhắc văn bản. Các mô hình hiện đại như `gpt-image` học mối quan hệ giữa văn bản và hình ảnh trong quá trình huấn luyện, sau đó lặp đi lặp lại biến nhiễu ngẫu nhiên thành một hình ảnh phù hợp với lời nhắc của bạn.

Hai dòng mô hình hình ảnh nổi tiếng là:

- **`gpt-image` (OpenAI)** — thế hệ hiện tại được sử dụng trong bài học này. Nó hỗ trợ tạo ảnh từ văn bản và chỉnh sửa ảnh (chèn ảnh với mặt nạ).
- **Midjourney** — một mô hình bên thứ ba phổ biến với dịch vụ và quy trình làm việc dựa trên Discord riêng.

> Các mô hình ảnh OpenAI cũ hơn — **DALL·E 2** và **DALL·E 3** — là kế thừa. DALL·E 3 không còn có sẵn cho các triển khai mới, và các tính năng như `create_variation` chỉ tồn tại trong DALL·E 2. Sử dụng các mô hình `gpt-image` cho các ứng dụng mới.

> **Quan trọng:** Các mô hình `gpt-image` trả về hình ảnh được tạo dưới dạng **base64** (`b64_json`), không phải dưới dạng URL. Mã của bạn sẽ giải mã chuỗi base64 thành byte và lưu lại — không có URL ảnh để tải xuống.


## Xây dựng ứng dụng tạo hình ảnh đầu tiên của bạn

Vậy để xây dựng một ứng dụng tạo hình ảnh cần những gì? Bạn cần các thư viện sau:

- **python-dotenv**, bạn được khuyên dùng thư viện này để giữ các bí mật trong một tệp *.env* tách biệt khỏi mã nguồn.
- **openai**, thư viện này sẽ dùng để tương tác với API của OpenAI.
- **pillow**, để làm việc với hình ảnh trong Python.


1. Tạo một tệp *.env* với nội dung sau:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Thu thập các thư viện ở trên vào một tập tin gọi là *requirements.txt* như sau:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Tiếp theo, tạo môi trường ảo và cài đặt các thư viện:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use the following commands to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add the following code in file called *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Let's explain this code:

- First, we import the libraries we need, including the OpenAI library, the dotenv library, the base64 module, and the Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- After that, we create the client, which reads the API key from your ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Next, we generate the image:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models return the image as a **base64** string in `data[0].b64_json`. We decode it to bytes and write it to a file — there's no URL to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lastly, we open the image and use the standard image viewer to display it:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### More details on generating the image

Let's look at the parameters of `images.generate`:

- **model**, is the image model to use (for example `gpt-image-2`).
- **prompt**, is the text prompt used to generate the image. Here it's "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is the size of the generated image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, is the number of images generated. Here we generate one.

> Image models don't take a `temperature` parameter — that's a text-generation control. To get variety, call the API again; to reduce variety, make your prompt more specific.

## Additional capabilities of image generation

You've seen how to generate an image with a few lines of Python. `gpt-image` models can also **edit** an existing image. By providing an image, an optional **mask** (which marks the area to change), and a prompt, you can alter part of an image — for example, add a hat to our bunny.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

The base image contains just the rabbit; the final image has the hat on the rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
